In [3]:
import torch
import torch.nn as nn
import math

# Word-level tokenization
words = text.split()
vocab = list(set(words))
word_to_ix = {w:i for i,w in enumerate(vocab)}
ix_to_word = {i:w for w,i in word_to_ix.items()}

data = [word_to_ix[w] for w in words]

seq_length = 5
X, Y = [], []

for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length])
    Y.append(data[i+1:i+seq_length+1])

X = torch.tensor(X)
Y = torch.tensor(Y)

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)

        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2*i)/d_model)))
                if i+1 < d_model:
                    pe[pos, i+1] = math.cos(pos / (10000 ** ((2*(i+1))/d_model)))

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Transformer Model
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos(x)
        x = x.permute(1, 0, 2)
        x = self.transformer(x)
        x = x.permute(1, 0, 2)
        return self.fc(x)

model = TransformerModel(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# Training
for epoch in range(100):
    outputs = model(X)
    loss = criterion(outputs.view(-1, len(vocab)), Y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# Generate sequence
def generate_words(seed, length=10):
    model.eval()
    input_seq = [word_to_ix[w] for w in seed.split()]

    for _ in range(length):
        inp = torch.tensor(input_seq[-seq_length:]).unsqueeze(0)
        out = model(inp)
        probs = torch.softmax(out[0, -1], dim=0)
        idx = torch.multinomial(probs, 1).item()
        input_seq.append(idx)

    return ' '.join(ix_to_word[i] for i in input_seq)

print(generate_words("machine learning"))

/tmp/ipykernel_3280/235465564.py:48: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)


Epoch 0, Loss: 4.587370872497559
Epoch 20, Loss: 0.2056412696838379
Epoch 40, Loss: 0.02013109251856804
Epoch 60, Loss: 0.007366036530584097
Epoch 80, Loss: 0.004985451232641935
machine learning platforms use artificial intelligence. students benefit from intelligent tutoring systems.
